# Defining a custom Fusarium Head Blight indicator

This tutorial implements a weather-based **Fusarium head blight (FHB) conducive-days indicator** that is not available in xclim. It demonstrates the complete extension pattern for **earthkit-climate** in one notebook:

1. define an xarray-based compute function using lower-level xclim machinery;
2. wrap it in an xclim `Indicator` instance;
3. expose it through an earthkit `format_handler` wrapper.

A day is FHB-conducive when all three conditions hold:

- mean temperature: 15 °C ≤ T ≤ 30 °C;
- relative humidity: RH ≥ 90%;
- daily precipitation: P > 0 mm.

The output is the yearly number of days satisfying all three conditions. This is a transparent weather-conduciveness indicator, not a calibrated disease probability; operational disease risk also depends on crop stage, cultivar susceptibility, inoculum, and local calibration.

In [1]:
from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd
import xarray as xr
from earthkit.utils.decorators import format_handler
from xclim.core.indicator import Daily
from xclim.core.units import convert_units_to, declare_units, rate2amount, to_agg_units

## 1. Define the compute function

The compute function owns the scientific definition. xclim is used only for unit-safe conversion, daily precipitation amount conversion, and aggregation units. The three threshold masks are combined with logical AND, so every condition must hold on a qualifying day.

In [2]:
@declare_units(
    tas="[temperature]",
    hurs="[]",
    pr="[precipitation]",
    tas_lower="[temperature]",
    tas_upper="[temperature]",
    hurs_lower="[]",
    pr_lower="[length]",
)
def fusarium_head_blight_conducive_days_index(
    tas: xr.DataArray,
    hurs: xr.DataArray,
    pr: xr.DataArray,
    tas_lower: str = "15 degC",
    tas_upper: str = "30 degC",
    hurs_lower: str = "90 %",
    pr_lower: str = "0 mm",
    freq: str = "YS",
) -> xr.DataArray:
    """Return the number of Fusarium head blight conducive days.

    A day is conducive when mean temperature is within the closed interval
    [tas_lower, tas_upper], relative humidity is at least hurs_lower, and
    daily precipitation is strictly greater than pr_lower.

    Parameters
    ----------
    tas : xarray.DataArray
        Daily mean near-surface air temperature.
    hurs : xarray.DataArray
        Daily mean relative humidity.
    pr : xarray.DataArray
        Daily precipitation flux or rate.
    tas_lower : str
        Inclusive lower temperature threshold.
    tas_upper : str
        Inclusive upper temperature threshold.
    hurs_lower : str
        Inclusive lower relative-humidity threshold.
    pr_lower : str
        Exclusive lower daily precipitation threshold.
    freq : str
        Aggregation frequency; ``YS`` produces one result per calendar year.

    Returns
    -------
    xarray.DataArray
        Number of FHB-conducive days for each period.
    """
    tas_c = convert_units_to(tas, "degC")
    hurs_pct = convert_units_to(hurs, "%")
    pr_amount = rate2amount(pr, out_units="mm")

    lower_t = convert_units_to(tas_lower, tas_c)
    upper_t = convert_units_to(tas_upper, tas_c)
    lower_rh = convert_units_to(hurs_lower, hurs_pct)
    lower_pr = convert_units_to(pr_lower, pr_amount)

    conducive = (tas_c >= lower_t) & (tas_c <= upper_t) & (hurs_pct >= lower_rh) & (pr_amount > lower_pr)

    out = conducive.resample(time=freq).sum(dim="time")
    out = to_agg_units(out, tas, op="count")
    return out.rename("fhb_conducive_days")

## 2. Create the xclim Indicator

The xclim layer adds daily-frequency validation, missing-value handling, dataset variable lookup, and output metadata. We instantiate xclim's existing `Daily` base because this single indicator does not need a new class with shared validation behaviour.

In [3]:
fhb_conducive_days_xclim = Daily(
    realm="atmos",
    identifier="fhb_conducive_days",
    compute=fusarium_head_blight_conducive_days_index,
    title="Fusarium head blight conducive days",
    abstract=("Yearly number of wet, very humid days with temperatures conducive to Fusarium head blight."),
    var_name="fhb_conducive_days",
    long_name="Fusarium head blight conducive days",
    description=(
        "Number of days where {tas_lower} <= T <= {tas_upper}, relative humidity "
        ">= {hurs_lower}, and precipitation > {pr_lower}."
    ),
    units="d",
    cell_methods="time: sum over days",
)

## 3. Add the earthkit format wrapper

The public wrapper follows the existing earthkit-climate API. Each climate variable can be supplied as a `DataArray` or resolved by name from `ds`, while `format_handler` enables compatible earthkit inputs to be translated to xarray.

In [4]:
@format_handler()
def fusarium_head_blight_conducive_days(
    tas: xr.DataArray | str = "tas",
    hurs: xr.DataArray | str = "hurs",
    pr: xr.DataArray | str = "pr",
    ds: xr.Dataset | Any = None,
    *,
    tas_lower: Any = "15 degC",
    tas_upper: Any = "30 degC",
    hurs_lower: Any = "90 %",
    pr_lower: Any = "0 mm",
    freq: str = "YS",
    **kwargs: Any,
) -> xr.DataArray:
    """Compute yearly Fusarium head blight conducive days."""
    return fhb_conducive_days_xclim(
        tas=tas,
        hurs=hurs,
        pr=pr,
        ds=ds,
        tas_lower=tas_lower,
        tas_upper=tas_upper,
        hurs_lower=hurs_lower,
        pr_lower=pr_lower,
        freq=freq,
        **kwargs,
    )

## 4. Build deterministic sample data

The synthetic dataset contains two complete years of daily mean temperature, relative humidity, and precipitation. It includes qualifying days and near misses where exactly one of the three conditions fails, making the logical-AND combiner directly testable.

In [5]:
time = pd.date_range("2020-01-01", "2021-12-31", freq="D")
shape = time.size

tas = xr.DataArray(
    np.full(shape, 20.0),
    coords={"time": time},
    dims="time",
    name="tas",
    attrs={
        "units": "degC",
        "standard_name": "air_temperature",
        "cell_methods": "time: mean within days",
    },
)
hurs = xr.DataArray(
    np.full(shape, 70.0),
    coords={"time": time},
    dims="time",
    name="hurs",
    attrs={
        "units": "%",
        "standard_name": "relative_humidity",
        "cell_methods": "time: mean within days",
    },
)
pr = xr.DataArray(
    np.zeros(shape),
    coords={"time": time},
    dims="time",
    name="pr",
    attrs={
        "units": "mm/day",
        "standard_name": "precipitation_flux",
        "cell_methods": "time: mean within days",
    },
)

# Days satisfying all three FHB conditions.
fhb_dates_2020 = ["2020-06-10", "2020-06-11", "2020-06-20"]
fhb_dates_2021 = ["2021-05-15", "2021-05-16", "2021-05-17", "2021-05-30"]
fhb_dates = fhb_dates_2020 + fhb_dates_2021
hurs.loc[{"time": fhb_dates}] = 95.0
pr.loc[{"time": fhb_dates}] = 1.0

# Near misses demonstrate that the combiner is ALL.
near_miss_dates = ["2020-06-25", "2021-06-01", "2021-06-02"]
hurs.loc[{"time": near_miss_dates}] = 95.0
pr.loc[{"time": near_miss_dates}] = 1.0
tas.loc[{"time": "2020-06-25"}] = 31.0  # Temperature above the upper bound.
hurs.loc[{"time": "2021-06-01"}] = 85.0  # Humidity below the lower bound.
pr.loc[{"time": "2021-06-02"}] = 0.0  # No precipitation.

ds = xr.Dataset({"tas": tas, "hurs": hurs, "pr": pr})
ds.sel(time=fhb_dates + near_miss_dates).to_dataframe()

,tas,hurs,pr
time,,,
2020-06-10,20.0,95.0,1.0
2020-06-11,20.0,95.0,1.0
2020-06-20,20.0,95.0,1.0
2021-05-15,20.0,95.0,1.0
2021-05-16,20.0,95.0,1.0
2021-05-17,20.0,95.0,1.0
2021-05-30,20.0,95.0,1.0
2020-06-25,31.0,95.0,1.0
2021-06-01,20.0,85.0,1.0


## 5. Compute and verify yearly FHB-conducive days

Every day satisfying all three conditions is counted: three days in 2020 and four in 2021. The near misses are excluded. The assertions make these expected values part of the executable tutorial.

In [6]:
fhb_days = fusarium_head_blight_conducive_days(ds=ds)

np.testing.assert_array_equal(fhb_days.values, [3, 4])
np.testing.assert_array_equal(fhb_days.time.dt.year.values, [2020, 2021])
assert fhb_days.attrs["units"] == "d"

fhb_days.to_dataframe()

,fhb_conducive_days
time,
2020-01-01,3.0
2021-01-01,4.0


The indicator is independent of xclim's published indicator catalogue: the threshold combination and FHB interpretation are defined in our compute function, while xclim supplies reusable unit, validation, missing-data, and metadata infrastructure. Before operational use, the thresholds and relevant crop-stage period should be validated for the target cultivar, pathogen population, and region.